# Aprendizado Não Supervisionado: Encontrando Padrões Ocultos
**Autores:** Jean Carlos, João Guilherme e Samuel Allan

Imagine que recebemos um banco de dados com o histórico de 100.000 usuários de um e-commerce. Nosso objetivo é criar campanhas de marketing personalizadas. O problema? Não temos rótulos (gabaritos) informando quem é o cliente "Frequente", "Casual" ou "Em Risco". 

É aqui que entra o **Aprendizado Não Supervisionado**. Como não temos o "gabarito", o modelo não otimiza acertos, mas sim **métricas de similaridade**, agrupando os dados de forma autônoma.

**Principais Aplicações:**
1. Agrupamento (Clustering)
2. Redução de Dimensionalidade
3. Detecção de Anomalias

## Configuração do Ambiente e Instalação
Antes de executar as análises, é recomendável criar um ambiente virtual para organizar as dependências do projeto. Se estiver executando localmente, abra o terminal e rode os comandos abaixo:

**No Windows:**
```bash
python -m venv venv
venv\Scripts\activate
```

**No Linux / macOS:**
```bash
python3 -m venv venv
source venv/bin/activate
```

In [ ]:
# Instalação das bibliotecas necessárias


!pip install scikit-learn matplotlib

In [ ]:
# Importando as dependências necessárias
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore') # Apenas para manter o notebook limpo na apresentação

## O Algoritmo K-Means na Prática
O K-Means agrupa os dados baseando-se na distância entre eles. O objetivo é garantir que os clientes de um mesmo grupo sejam o mais parecidos possível (minimizando a inércia).

Passos do algoritmo:
1. **Inicialização:** Escolhe K centros aleatórios (centroides).
2. **Atribuição:** Cada cliente é associado ao centroide mais próximo.
3. **Atualização:** O centroide é movido para o "meio" exato de seu grupo.
4. **Convergência:** Repete-se até os centroides pararem de se mover.

Vamos simular e visualizar essa extração matemática criando nossa própria base de clientes.

In [ ]:
# 1. Geração da nossa base de clientes (300 clientes simulados, divididos em 4 perfis originais)
X, y_verdadeiro = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=0)

# 2. Instanciando nosso algoritmo
kmeans = KMeans(n_clusters=4, n_init='auto', random_state=42)

# 3. Fit (Treinamento): o algoritmo se ajusta e agrupa iterativamente
kmeans.fit(X)

# 4. Extração dos perfis (grupos) finais e onde estão os líderes (centroides)
perfis_identificados = kmeans.labels_
centroides = kmeans.cluster_centers_

# Visualização da mágica acontecendo
plt.figure(figsize=(10, 6))

# Associamos o plot principal à variável 'grafico' para extrair os elementos da legenda depois
grafico = plt.scatter(X[:, 0], X[:, 1], c=perfis_identificados, cmap='viridis', alpha=0.6, s=50)

# Plotamos os centroides separadamente e guardamos em 'plot_centroides'
plot_centroides = plt.scatter(centroides[:, 0], centroides[:, 1], c='red', marker='X', s=200, label='Centroides')

plt.title("Perfis de Clientes Segmentados via K-Means", fontsize=14)
plt.xlabel("Frequência de Visitas (Exemplo)")
plt.ylabel("Valor Gasto (Exemplo)")

# Boas práticas de mercado: Unindo as labels de negócio com a label técnica do centroide
handles = grafico.legend_elements()[0]
handles.append(plot_centroides)
labels = ['Campanha A', 'Campanha B', 'Campanha C', 'Campanha D', 'Centroides']

plt.legend(handles=handles, labels=labels, title="Ações de Marketing", loc='upper right')

plt.grid(True, alpha=0.3)
plt.show()

## Mitigando Erros Comuns: Como descobrir o "K" ideal?
No algoritmo K-Means puro, precisamos definir quantos grupos queremos (o parâmetro escalar *K*). Se chutarmos um número errado, o algoritmo forçará quebras que não têm significado de negócios. 

Para resolver isso, usamos o **Método do Cotovelo (Elbow Method)**, que testa o treinamento várias vezes, com diferentes números de grupos, e nos ajuda a encontrar o ponto de equilíbrio matemático (onde a inércia para de cair drasticamente).

In [ ]:
inercia = []
valores_k = range(1, 11)

# Testando o K-Means de 1 até 10 grupos diferentes
for k in valores_k:
    modelo_teste = KMeans(n_clusters=k, n_init='auto', random_state=42)
    modelo_teste.fit(X)
    inercia.append(modelo_teste.inertia_) # Salvando o quão "espalhado" o grupo ficou

# Plotando o Gráfico do Cotovelo
plt.figure(figsize=(8, 5))
plt.plot(valores_k, inercia, marker='o', linestyle='--', color='b')
plt.title('Método do Cotovelo (Elbow Method)', fontsize=14)
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia (Distância residual)')
plt.xticks(valores_k)
plt.axvline(x=4, color='red', linestyle=':', label='K Ideal (O Cotovelo)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Conclusão visual: O gráfico faz uma curva aguda no K=4 e depois estabiliza!

## Outro ponto crítico: A Sensibilidade à Escala
Como o K-Means calcula distâncias geométricas, ele é "cego" às diferentes escalas numéricas. Misturar *Idade (20 a 80 anos)* com *Renda Mensal (R$ 1.000 a R$ 50.000)* fará a distância da renda esmagar a idade matematicamente.

**Solução:** Sempre passar os dados por um padronizador antes do treino, como o `StandardScaler` do `scikit-learn`.